In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, f1_score, recall_score
import joblib
import mlflow
import mlflow.sklearn

In [2]:
mlflow.set_tracking_uri("https://mlflow.cpetit.fr")
mlflow.set_experiment("Titanic ML experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1765532571343, experiment_id='1', last_update_time=1765532571343, lifecycle_stage='active', name='Titanic ML experiment', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:
df = pd.read_csv("data/titanic.csv")

In [4]:
X = df[['Sex', 'Pclass', 'Age']].copy()  # Genre et classe du ticket
y = df['Survived']          # Variable cible

# Encoder la variable 'Sex' (male/female -> 0/1)
le = LabelEncoder()
X['Sex'] = le.fit_transform(X['Sex'])

X_clean = X.dropna()
y_clean = y[X_clean.index]
test_size = 0.2

# Split stratifié (important car le taux de survie est déséquilibré)
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean,
    test_size=test_size,        # 80% train, 20% test
    random_state=42,      # reproductibilité
    stratify=y_clean            # préserve le ratio survivants/non-survivants
)

print(f"Taille train: {len(X_train)}")
print(f"Taille test: {len(X_test)}")
print(f"Distribution train: {y_train.value_counts(normalize=True)}")
print(f"Distribution test: {y_test.value_counts(normalize=True)}")

Taille train: 571
Taille test: 143
Distribution train: Survived
0    0.593695
1    0.406305
Name: proportion, dtype: float64
Distribution test: Survived
0    0.594406
1    0.405594
Name: proportion, dtype: float64


In [5]:
with mlflow.start_run(run_name="linear_regression_v1") as run:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(solver="liblinear", class_weight="balanced")
    model.fit(X_train_scaled, y_train)

    # Prédictions sur train et test
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    # Matrice de confusion pour le test set
    test_conf_matrix = confusion_matrix(y_test, y_test_pred)

    fig = plt.figure(figsize=(8,6))
    sns.heatmap(test_conf_matrix, annot=True, fmt='d', cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Reality")
    plt.title("Test Confusion matrix")

    # Log de la matrice de confusion
    mlflow.log_figure(fig, "test_confusion_matrix.png")
    plt.close()

    # Matrice de confusion pour le train set
    train_conf_matrix = confusion_matrix(y_train, y_train_pred)

    fig = plt.figure(figsize=(8,6))
    sns.heatmap(train_conf_matrix, annot=True, fmt='d', cmap='Blues')
    plt.xlabel("Predicted")
    plt.ylabel("Reality")
    plt.title("Train Confusion matrix")

    # Log de la matrice de confusion
    mlflow.log_figure(fig, "train_confusion_matrix.png")
    plt.close()

    # Log des paramètres
    mlflow.log_param("test_size", test_size)

    # Log des métriques TRAIN SET
    mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
    mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
    mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
    mlflow.log_metric("train_f1_score", f1_score(y_train, y_train_pred))

    # Log des métriques TEST SET
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1_score", f1_score(y_test, y_test_pred))

    # Enregistrement du modèle
    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="linear_regression_model"
    )

    print(f"\nRun ID: {run.info.run_id}")
    print(f"Model URI: runs:/{run.info.run_id}/model")

2025/12/12 14:32:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'linear_regression_model' already exists. Creating a new version of this model...
2025/12/12 14:32:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: linear_regression_model, version 6
Created version '6' of model 'linear_regression_model'.



Run ID: af9749b44e5c4d15806464ec9ecae023
Model URI: runs:/af9749b44e5c4d15806464ec9ecae023/model
🏃 View run linear_regression_v1 at: https://mlflow.cpetit.fr/#/experiments/1/runs/af9749b44e5c4d15806464ec9ecae023
🧪 View experiment at: https://mlflow.cpetit.fr/#/experiments/1


In [6]:
loaded_model = mlflow.sklearn.load_model(f"runs:/{run.info.run_id}/model")

# Test de prédiction
test_sample = X_test_scaled[:5]
predictions = loaded_model.predict(test_sample)

print("\n=== Test de prédiction ===")
print(f"Prédictions: {predictions}")
print(f"Vraies valeurs: {y_test[:5]}")

d:\Suivi de formation\simplon-ai-developer-training\W29-ML-Monitoring\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Test de prédiction ===
Prédictions: [1 0 0 0 0]
Vraies valeurs: 315    1
483    1
736    0
722    0
400    1
Name: Survived, dtype: int64


In [7]:
joblib.dump(model, "models/ml_model.pkl")
joblib.dump(scaler, "models/ml_scaler.pkl")

['models/ml_scaler.pkl']